## Importing modules

In [1]:
import numpy as np
import pandas as pd

### Importing from files

In [2]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Now import the module
from relativistic_dof import RelativisticDOFRegistry

## Comparison between different DOF models in axion production

In [3]:
# --- Set cases of our interest
# Partciles masses
taon_mass = 1777 * 10**6   # [eV]
muon_mass = 105.66*10**6   # [eV]
electron_mass = 511*10**3  # [eV]

# Set decouple moment
x_dec = 30  # decouple from the plasma

# Define axion production parameters
axion_production = {
    "taon": {
        "mass": taon_mass,
        "x_dec": x_dec
    },
    "muon": {
        "mass": muon_mass,
        "x_dec": x_dec
    },
    "electron": {
        "mass": electron_mass,
        "x_dec": x_dec
    }
}

In [4]:
# --------------------------------------- PREDICTION OF TABLE METHOD --------------------------------------- #
# Set the class
RelativisticDOF = RelativisticDOFRegistry.get_method("table")

for key in axion_production.keys():
    # Calculate DOF
    axion_decouple_gS = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_s")
    axion_decouple_gE = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_e")

    # update the dictionary
    axion_production[key]["table"] = dict()
    axion_production[key]["table"]["g_s"] = axion_decouple_gS
    axion_production[key]["table"]["g_e"] = axion_decouple_gE
    
# --------------------------------------- PREDICTION OF FIT METHOD --------------------------------------- #
# Set the class
RelativisticDOF = RelativisticDOFRegistry.get_method("fit")

for key in axion_production.keys():
    # Calculate DOF
    axion_decouple_gS = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_s")
    axion_decouple_gE = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_e")

    # update the dictionary
    axion_production[key]["fit"] = dict()
    axion_production[key]["fit"]["g_s"] = axion_decouple_gS
    axion_production[key]["fit"]["g_e"] = axion_decouple_gE

#--------------------------------------- PREDICTION OF LATTICE MODEL --------------------------------------- #
# Set the class
RelativisticDOF = RelativisticDOFRegistry.get_method("lattice")

for key in axion_production.keys():
    # Calculate DOF
    axion_decouple_gS = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_s")
    axion_decouple_gE = RelativisticDOF.compute_decoupling_dof(axion_production[key]["mass"], 
                                                               axion_production[key]["x_dec"],
                                                               dof_type="g_eff_e")

    # update the dictionary
    axion_production[key]["lattice"] = dict()
    axion_production[key]["lattice"]["g_s"] = axion_decouple_gS
    axion_production[key]["lattice"]["g_e"] = axion_decouple_gE

In [5]:
# Create a structured table for comparison
results = []

for particle, data in axion_production.items():
    row = {
        "Particle": particle.capitalize(),
        "Mass [eV]": data["mass"],
        "x_dec": data["x_dec"],
        "g_s [Table]": data["table"]["g_s"],
        "g_e [Table]": data["table"]["g_e"],
        "g_s [Fit]": data["fit"]["g_s"],
        "g_e [Fit]": data["fit"]["g_e"],
        "g_s [Lattice]": data["lattice"]["g_s"],
        "g_e [Lattice]": data["lattice"]["g_e"],
    }
    results.append(row)

# Convert to DataFrame for easy display
df_results = pd.DataFrame(results)

In [6]:
df_results

,Particle,Mass [eV],x_dec,g_s [Table],g_e [Table],g_s [Fit],g_e [Fit],g_s [Lattice],g_e [Lattice]
0,Taon,1.777000e+09,30,14.989433,15.330851,14.933294,15.247789,14.690158,15.024809
1,Muon,1.056600e+08,30,10.728420,10.730941,10.718468,10.716604,10.738716,10.741822
2,Electron,5.110000e+05,30,3.910000,3.360000,3.931000,3.383000,10.663646,10.696369


### Cross check

In [17]:
RelativisticDOF = RelativisticDOFRegistry.get_method("fit")

low_kB_T = 1e1
today_gS = RelativisticDOF.get_degrees_of_freedom(low_kB_T, dof_type="g_eff_s")

print("today DOF (entropy):", today_gS)

today DOF (entropy): 3.931


In [18]:
# ---
print("Nie wiem dlaczego, ale u Maxima w przypadku taon decays DOF entropii powinno być: 15.4185")
print("Nie wiem który model używał podczas obliczania Neff")

Nie wiem dlaczego, ale u Maxima w przypadku taon decays DOF entropii powinno być: 15.4185
Nie wiem który model używał podczas obliczania Neff
